# HW3 - PEFT

In this notebook, we will fine-tune the GPT2 model on the [WikiText](https://huggingface.co/datasets/Salesforce/wikitext#wikitext-2-v1) dataset using different fine-tuning methodologies.

Parameter-Efficient Fine-Tuning (PEFT) is a technique that enables the adaptation of large pre-trained models to specific tasks while modifying only a small subset of their parameters, significantly reducing computational and memory costs. Instead of updating all model parameters, PEFT methods, such as LoRA (Low-Rank Adaptation), Adapter layers, and Prefix-Tuning, introduce lightweight trainable modules that are inserted into the model or modify activations in a structured way. This approach retains the general knowledge of the base model while efficiently adapting to new tasks, making it particularly useful for fine-tuning large-scale models like LLMs and vision-language models on resource-constrained hardware.

## Install required libraries

In [60]:
pip install datasets

## Import required libraries

In [61]:
pip install peft

In [62]:
import gc
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
!pip install -U "torchao>0.16.0"
from peft import LoraConfig, PrefixTuningConfig, get_peft_model, PeftModel

In [63]:
import torch
import torchao
import peft

print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)
print("PEFT:", peft.__version__)

PyTorch: 2.11.0+cu128
TorchAO: 0.18.0
PEFT: 0.19.1


## Directory Roots

In [64]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls "/content/drive/MyDrive"

'Colab Notebooks'  'Deep Learning - SUT'


In [66]:
BASE_ROOT = "/content/drive/MyDrive/Deep Learning - SUT/3 - HW3"
import os

print(os.listdir(BASE_ROOT))


['data', 'Models', 'Results', 'Runs']


In [67]:
data_root = f"{BASE_ROOT}/data/Q3_PEFT/"
Models_root = f"{BASE_ROOT}/Models/Q3_PEFT/"
Results_root = f"{BASE_ROOT}/Results/Q3_PEFT/"
Runs_root = f"{BASE_ROOT}/Runs/Q3_PEFT/"

### device

In [68]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


In [69]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

cuda


## Setup

In [70]:
gpt_2_medium_model_name = "openai-community/gpt2-medium"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(gpt_2_medium_model_name)
tokenizer.pad_token = tokenizer.eos_token

# Tokenize the dataset
def tokenizing_preprocess(examples):
    inputs =  tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)
    inputs['labels'] = inputs['input_ids'].copy()
    return inputs


# Define training arguments
training_args = TrainingArguments(
    output_dir='./gpt2',
    eval_strategy='no',
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    report_to="none"
)

## Load dataset (5 pt)

In [71]:
# TODO: Load the wikitext-2-v1 version of wikitext
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-v1")

### saving the dataset

In [72]:
dataset.save_to_disk(data_root + "/wikitext-2-v1")

Saving the dataset (0/1 shards):   0%|          | 0/4358 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/36718 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/3760 [00:00<?, ? examples/s]

In [73]:
dataset

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [74]:
dataset["train"][10]

{'text': ' The game \'s battle system , the <unk> system , is carried over directly from <unk> Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player moves the character around the battlefield in third @-@ person . A character can only act once per @-@ turn , but characters can be granted multiple turns at the expense of other characters \' turns . Each character has a field and distance of movement limited by their Action <unk> . Up to nine characters can be assigned to a single mission . During gameplay , characters will call out if something happens to them , such as their health points ( HP ) getting low or being knocked out by enemy attacks . Each character has specific " Potentials " , skills unique to each character . They are divided into " Personal Potential " , which are innate skills that remain unaltered unless otherwise dictated by the story and can either help or impede a 

In [75]:
# TODO Select 1000 data for train and 500 data for validation
train_data = dataset["train"].select(range(1000))
eval_data = dataset["validation"].select(range(500))

# Apply tokenization preprocess on datasets
train_dataset = train_data.map(tokenizing_preprocess, batched=True)
eval_dataset = eval_data.map(tokenizing_preprocess, batched=True)

In [76]:
train_dataset.save_to_disk(data_root + "training data")
eval_dataset.save_to_disk(data_root + "validation data")

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

### counting the trainable params for each method

In [77]:
def get_trainable_params(model):
    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return trainable_params / 1e6

## Full Fine-Tuning (5 pt)

In [78]:
# Load the model
ff_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

In [79]:
# Initialize Trainer
trainer = Trainer(
    model=ff_model,
    args=training_args,
    train_dataset=train_dataset,
)

> **Note**: a side note is that the ``Trainer`` method, automatially send the trainer into the gpu.

In [80]:
print(trainer.args.device)

cuda:0


In [81]:
# Zero-Shot evaluation of model

# TODO: Evaluate model on eval_dataset
ff_model.eval()
eval_output = trainer.evaluate(
    eval_dataset=eval_dataset
)

print(f"eval_loss = {eval_output['eval_loss']:.4f}")

Training Loss,Validation Loss,Step
No log,8.475959,0


eval_loss = 8.4760


In [82]:
# TODO: Get reserved memory from CUDA
gpu_memory_before = torch.cuda.memory_reserved(device)

# TODO: Train the model using trainer
train_output = trainer.train()

# TODO: Get reserved memory from CUDA
gpu_memory_after = torch.cuda.memory_reserved(device)

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")
print(f"GPU memory used: {(gpu_memory_after - gpu_memory_before)/(10**9):.4f} GBytes")

Step,Training Loss


Training time: 117.5038 seconds
GPU memory used: 5052039168.0000 bytes
GPU memory used: 5.0520 GBytes


In [83]:
# TODO: Evaluate model on eval_dataset
ff_model.eval()
eval_output = trainer.evaluate(
    eval_dataset=eval_dataset
)
print(f"eval_loss = {eval_output['eval_loss']:.4f}")

Training Loss,Validation Loss,Step
No log,1.067852,250


eval_loss = 1.0679


In [84]:
full_ft_params = get_trainable_params(ff_model)
print(f"Full Fine-Tuning: {full_ft_params:.3f} M")

Full Fine-Tuning: 354.823 M


In [85]:
# Delete the model
del ff_model
del trainer

In [86]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close to zero(~0.2))
gc.collect()
torch.cuda.empty_cache()

## Prefix Tuning (20 pt)

TODO: Explain about Prefix Tuning briefly

In [87]:
from transformers import AutoModel
prefix_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

In [88]:
# TODO: Define your LoRA configuration using PrefixTuningConfig class from peft library
#       Set task_type to CAUSAL_LM

from peft import PrefixTuningConfig, get_peft_model, TaskType

prefix_config = PrefixTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=20
)

# TODO: Wrraped the GPT2LMHeadModel with above prefix config using get_peft_model function
prefix_model = get_peft_model(
    prefix_model,
    prefix_config
)
# TODO: Print number of trainable parameters
prefix_model.print_trainable_parameters()

trainable params: 983,040 || all params: 355,806,208 || trainable%: 0.2763


In [89]:
prefix_model

PeftModelForCausalLM(
  (base_model): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(50257, 1024)
      (wpe): Embedding(1024, 1024)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-23): 24 x GPT2Block(
          (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D(nf=3072, nx=1024)
            (c_proj): Conv1D(nf=1024, nx=1024)
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): GPT2MLP(
            (c_fc): Conv1D(nf=4096, nx=1024)
            (c_proj): Conv1D(nf=1024, nx=4096)
            (act): NewGELUActivation()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    )
    (lm_head): Linear(in_fea

In [90]:
# Initialize Trainer
trainer = Trainer(
    model=prefix_model,
    args=training_args,
    train_dataset=train_dataset,
)

In [91]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved(device)
# TODO: Train the model
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved(device)

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")
print(f"GPU memory used: {(gpu_memory_after - gpu_memory_before)/(10**9):.4f} GBytes")

Step,Training Loss


Training time: 74.7539 seconds
GPU memory used: 1646264320.0000 bytes
GPU memory used: 1.6463 GBytes


In [92]:
# TODO: Evaluate model on eval_dataset
prefix_model.eval()
eval_output = trainer.evaluate(
    eval_dataset=eval_dataset
)
print(f"eval_loss = {eval_output['eval_loss']:.4f}")

Training Loss,Validation Loss,Step
No log,9.451506,250


eval_loss = 9.4515


In [93]:
full_ft_params = get_trainable_params(prefix_model)
print(f"Full Fine-Tuning: {full_ft_params:.3f} M")

Full Fine-Tuning: 0.983 M


In [94]:
# Delete the model
del prefix_model
del trainer

In [95]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close or less than 1.5Gb)
gc.collect()
torch.cuda.empty_cache()

## Fine-Tuning by LoRA (Low-Rank Adaptation) (40 pt)

TODO: Explain about LoRA (Low-Rank Adaptation) briefly

In [96]:
lora_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

In [97]:
# Print the model artitechture
print(lora_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)


In [98]:
# TODO: Define your LoRA configuration using LoraConfig class from peft library
#       Apply the LoRA on Conv1D modules (c_attn and c_proj) of GPT2Attention blocks (attn).
#       Set fan_in_fan_out to True
#       Set task_type to CAUSAL_LM
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,    # rank of the matrix
    lora_alpha=16,  # Scaling
    lora_dropout=0.1,
    target_modules=["c_attn", "c_proj"],    #impact on the layers
    fan_in_fan_out=True,
    task_type=TaskType.CAUSAL_LM
)

# # TODO: Wrraped the transformer module of GPT2LMHeadModel with above lora config
# #       using get_peft_model function
lora_model = get_peft_model(
    lora_model,
    lora_config
)

# TODO: Print number of trainable parameters
lora_model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 356,985,856 || trainable%: 0.6058


In [99]:
# Print the model artitechture and see the changes
print(lora_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 1024)
        (wpe): Embedding(1024, 1024)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-23): 24 x GPT2Block(
            (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=3072, nx=1024)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
      

In [100]:
# Initialize Trainer
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
)

In [101]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved(device)
# TODO: Train the model using trainer
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved(device)

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")
print(f"GPU memory used: {(gpu_memory_after - gpu_memory_before)/(10**9):.4f} GBytes")

Step,Training Loss


Training time: 80.2606 seconds
GPU memory used: 1920991232.0000 bytes
GPU memory used: 1.9210 GBytes


In [102]:
# TODO: Evaluate model on eval_dataset
lora_model.eval()
eval_output = trainer.evaluate(
    eval_dataset=eval_dataset
)
print(f"eval_loss = {eval_output['eval_loss']:.4f}")

Training Loss,Validation Loss,Step
No log,6.538172,250


eval_loss = 6.5382


In [103]:
full_ft_params = get_trainable_params(lora_model)
print(f"Full Fine-Tuning: {full_ft_params:.3f} M")

Full Fine-Tuning: 2.163 M


In [104]:
# Delete the model
del lora_model
del trainer

In [105]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close or less than 1.5Gb)
gc.collect()
torch.cuda.empty_cache()

#### Run LoRA for different rank values

Fine-tune the GPT-2 model with different rank values. (Be sure to change the alpha value according to the rank so that the results are fair.)

Enter the requested items in the table.

Compare the values ​​obtained and explain their differences.

TODO

| Method | Training Time(s) | Training Memory(Gb) | Validation Loss| #Trainable Params(M)|
|:-:|:-:|:-:|:-:|:-:|
| Zero-Shot         | NaN  | NaN  | 8.4760 | NaN |
| Full Fine-Tuning  | 117.5038  | 5.0520  | 1.0679 | 354.823 |
| Prefix Tuning     | 74.7539  | 1.6463  | 9.4951 | 0.983 |
| Lora rank=4       | 79.9910  | 1.912603  | 7.414558 | 1.081344 |
| Lora rank=16      | 80.2370  | 1.939866  | 4.675036 | 4.325376 |
| Lora rank=64      | 82.5561  | 2.059403  | 1.346176 | 17.301504 |
| Lora rank=256     | 94.9293  | 2.449474  | 1.221619 | 69.206016 |




TODO:

Your detailed and complete explanation

##### LoRA Loop

In [106]:
from peft import LoraConfig, get_peft_model, TaskType

In [107]:
ranks_list = [4, 16, 64, 256]
lora_alpha_list = [r * 2 for r in ranks_list]

evaluation_criterias = {}

for row, (r, alpha) in enumerate(
    zip(ranks_list, lora_alpha_list),
    start=1
):

    # -----------------------------
    # Create fresh base model
    # -----------------------------
    lora_base_model = AutoModelForCausalLM.from_pretrained(
        gpt_2_medium_model_name
    ).to(device)

    # -----------------------------
    # LoRA config
    # -----------------------------
    lora_config = LoraConfig(
        r=r,
        lora_alpha=alpha,
        lora_dropout=0.1,
        target_modules=["c_attn", "c_proj"],
        fan_in_fan_out=True,
        task_type=TaskType.CAUSAL_LM
    )

    # -----------------------------
    # Apply LoRA
    # -----------------------------
    lora_model = get_peft_model(
        lora_base_model,
        lora_config
    )

    # -----------------------------
    # Trainer
    # -----------------------------
    trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )

    # -----------------------------
    # Train
    # -----------------------------
    gpu_memory_before = torch.cuda.memory_reserved(device)

    train_output = trainer.train()

    gpu_memory_after = torch.cuda.memory_reserved(device)

    # -----------------------------
    # Evaluate
    # -----------------------------
    eval_output = trainer.evaluate()

    # -----------------------------
    # Save results FIRST
    # -----------------------------
    evaluation_criterias[r] = {
        "rank": r,
        "alpha": alpha,
        "training_time": train_output.metrics["train_runtime"],
        "gpu_memory_gb": (
            gpu_memory_after - gpu_memory_before
        ) / 10**9,
        "validation_loss": eval_output["eval_loss"],
        "trainable_params_m": get_trainable_params(lora_model)
    }

    print(evaluation_criterias[r])

    # -----------------------------
    # Delete model and trainer
    # -----------------------------
    del lora_model
    del lora_base_model
    del trainer

    # -----------------------------
    # Free GPU memory
    # -----------------------------
    gc.collect()
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Step,Training Loss


Training Loss,Validation Loss,Step
No log,7.414558,250


{'rank': 4, 'alpha': 8, 'training_time': 79.991, 'gpu_memory_gb': 1.912602624, 'validation_loss': 7.414558410644531, 'trainable_params_m': 1.081344}


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

...

{'rank': 64, 'alpha': 128, 'training_time': 82.5561, 'gpu_memory_gb': 2.059403264, 'validation_loss': 1.346176266670227, 'trainable_params_m': 17.301504}


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Step,Training Loss


Training Loss,Validation Loss,Step
No log,1.221619,250


{'rank': 256, 'alpha': 512, 'training_time': 94.9293, 'gpu_memory_gb': 2.449473536, 'validation_loss': 1.2216194868087769, 'trainable_params_m': 69.206016}


In [108]:
import pandas as pd

# Convert evaluation results to DataFrame

df_lora_results = pd.DataFrame.from_dict(
    evaluation_criterias,
    orient="index"
).reset_index(drop=True)

# Display results
display(df_lora_results)

# Save to Excel
excel_path = Results_root + "lora_rank_comparison.xlsx"

df_lora_results.to_excel(
    excel_path,
    index=False
)

print(f"Results saved to: {excel_path}")

,rank,alpha,training_time,gpu_memory_gb,validation_loss,trainable_params_m
0,4,8,79.9910,1.912603,7.414558,1.081344
1,16,32,80.2370,1.939866,4.675036,4.325376
2,64,128,82.5561,2.059403,1.346176,17.301504
3,256,512,94.9293,2.449474,1.221619,69.206016


Results saved to: /content/drive/MyDrive/Deep Learning - SUT/3 - HW3/Results/Q3_PEFT/lora_rank_comparison.xlsx


## Implement LoRA from scratch (30 pt)

In [109]:
custom_lora_model = AutoModelForCausalLM.from_pretrained(gpt_2_medium_model_name)
print(custom_lora_model)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=50257, bias=False)
)


In [110]:
class LoRALayer(nn.Module):
    def __init__(self, base_layer, rank=8, alpha=16):
        super().__init__()

        self.base_layer = base_layer
        self.rank = rank
        self.alpha = alpha

        # GPT-2 Conv1D:
        # weight shape = [in_features, out_features]
        self.in_features = base_layer.weight.shape[0]
        self.out_features = base_layer.weight.shape[1]

        # LoRA A: in_features -> rank
        self.lora_A = nn.Linear(
            self.in_features,
            self.rank,
            bias=False
        )

        # LoRA B: rank -> out_features
        self.lora_B = nn.Linear(
            self.rank,
            self.out_features,
            bias=True
        )

        # B must start from zero
        nn.init.zeros_(self.lora_B.weight)
        nn.init.zeros_(self.lora_B.bias)

        # Scaling factor
        self.scaling = self.alpha / self.rank

    def forward(self, x):

        # Original GPT-2 Conv1D
        base_output = self.base_layer(x)

        # LoRA path
        lora_output = self.lora_B(
            self.lora_A(x)
        )

        # Combine original + LoRA update
        return base_output + self.scaling * lora_output

In [111]:
# TODO: Freeze the model

for param in custom_lora_model.parameters():
    param.requires_grad = False

# TODO: Loop over list of GPT2Blocks of model and replace the Conv1D
#       modules (c_attn, c_proj) of them with your LoRALayer
for block in custom_lora_model.transformer.h:

    block.attn.c_attn = LoRALayer(
        block.attn.c_attn,
        rank=8,
        alpha=16
    )

    block.attn.c_proj = LoRALayer(
        block.attn.c_proj,
        rank=8,
        alpha=16
    )

In [112]:
print(custom_lora_model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): LoRALayer(
            (base_layer): Conv1D(nf=3072, nx=1024)
            (lora_A): Linear(in_features=1024, out_features=8, bias=False)
            (lora_B): Linear(in_features=8, out_features=3072, bias=True)
          )
          (c_proj): LoRALayer(
            (base_layer): Conv1D(nf=1024, nx=1024)
            (lora_A): Linear(in_features=1024, out_features=8, bias=False)
            (lora_B): Linear(in_features=8, out_features=1024, bias=True)
          )
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp)

In [113]:
# Initialize Trainer
trainer = Trainer(
    model=custom_lora_model,
    args=training_args,
    train_dataset=train_dataset,
)

In [114]:
# TODO: Get reserved memory from cuda
gpu_memory_before = torch.cuda.memory_reserved(device)
# TODO: Train the model using trainer
train_output = trainer.train()
# TODO: Get reserved memory from cuda
gpu_memory_after = torch.cuda.memory_reserved(device)

# Report the training time and gpu memory consumption
print(f"Training time: {train_output.metrics['train_runtime']:.4f} seconds")
print(f"GPU memory used: {gpu_memory_after - gpu_memory_before:.4f} bytes")
print(f"GPU memory used: {(gpu_memory_after - gpu_memory_before)/(10**9):.4f} GBytes")

Step,Training Loss


Training time: 75.7828 seconds
GPU memory used: 1574961152.0000 bytes
GPU memory used: 1.5750 GBytes


In [ ]:
# TODO: Evaluate model on eval_dataset
custom_lora_model.eval()
eval_output = trainer.evaluate(
    eval_dataset=eval_dataset
)
print(f"eval_loss = {eval_output['eval_loss']:.4f}")

In [ ]:
full_ft_params = get_trainable_params(custom_lora_model)
print(f"Full Fine-Tuning: {full_ft_params:.3f} M")

In [ ]:
# Delete the model
del custom_lora_model
del trainer

In [115]:
# Empty the GPU memory (Run this cell twice if the GPU RAM is not close or less than 1.5Gb)
gc.collect()
torch.cuda.empty_cache()